# 🛡️ Nullcon Security Analysis Pipeline

**Vulnerability Detection, Reachability Analysis & AI-Powered Remediation**

This notebook runs a complete security analysis pipeline:

1. **Clone** the nullcon-repo from GitHub
2. **Trivy** — Scan for HIGH/CRITICAL CVEs in dependencies
3. **Call Graph** — Build AST-based call graph for reachability analysis
4. **Reachability** — Use Groq LLM to identify which CVEs are actually used in code
5. **Fix Reachable** — LLM-generated fixes for reachable dependency vulnerabilities
6. **Bandit** — Static analysis for code-level security issues
7. **Fix Exploitable** — LLM-generated fixes for HIGH/CRITICAL Bandit findings

> **Prerequisites:** Groq API key from [console.groq.com](https://console.groq.com)

---
## Configuration

Set your Groq API key and target folder below.

In [1]:
import os
import sys
import subprocess
import shutil
from pathlib import Path

# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — Replace with your Groq API key
# ═══════════════════════════════════════════════════════════════
GROQ_API_KEY = ''  # <── replace with your key from console.groq.com

if 'YOUR_KEY' in GROQ_API_KEY:
    print('⚠️  WARNING: Replace GROQ_API_KEY with your actual key from console.groq.com')

REPO_URL = 'https://github.com/jp0402502-oss/nullcon-repo'
TARGET_FOLDER = 'sample1'  # Folder containing app.py and requirements.txt
WORK_DIR = Path('./nullcon-repo')  # Cloned repo location

# Set environment for child processes
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

---
## Step 1 — Clone the Repository

In [2]:
if WORK_DIR.exists():
    print(f'Removing existing {WORK_DIR}...')
    shutil.rmtree(WORK_DIR)

print(f'Cloning {REPO_URL}...')
result = subprocess.run(
    ['git', 'clone', REPO_URL, str(WORK_DIR)],
    capture_output=True, text=True, timeout=120
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Git clone failed')

os.chdir(WORK_DIR)
print(f'✅ Cloned to {WORK_DIR.absolute()}')

Cloning https://github.com/jp0402502-oss/nullcon-repo...
✅ Cloned to /content/nullcon-repo/nullcon-repo


---
## Step 2 — Install Trivy and Run CVE Scan

In [3]:
%%bash
# ── Install Trivy (Ubuntu / Google Colab) ──
if ! command -v trivy &> /dev/null; then
    echo "⏳ Installing Trivy..."
    sudo apt-get install -y wget apt-transport-https gnupg lsb-release > /dev/null 2>&1
    wget -qO - https://aquasecurity.github.io/trivy-repo/deb/public.key \
      | gpg --dearmor | sudo tee /usr/share/keyrings/trivy.gpg > /dev/null
    echo "deb [signed-by=/usr/share/keyrings/trivy.gpg] https://aquasecurity.github.io/trivy-repo/deb generic main" \
      | sudo tee /etc/apt/sources.list.d/trivy.list
    sudo apt-get update -qq > /dev/null 2>&1
    sudo apt-get install -y trivy > /dev/null 2>&1
    echo "✅ Trivy installed!"
else
    echo "✔ Trivy already installed"
fi
trivy --version

⏳ Installing Trivy...
deb [signed-by=/usr/share/keyrings/trivy.gpg] https://aquasecurity.github.io/trivy-repo/deb generic main
✅ Trivy installed!
Version: 0.69.1


In [4]:
%%bash
# ── Run Trivy: JSON output for the pipeline ──
trivy fs --severity HIGH,CRITICAL --format json --output ./report/trivy_results.json ./sample1
echo "✅ Saved to trivy_results.json"

✅ Saved to trivy_results.json


2026-02-27T17:13:18Z	INFO	[vulndb] Need to update DB
2026-02-27T17:13:18Z	INFO	[vulndb] Downloading vulnerability DB...
2026-02-27T17:13:18Z	INFO	[vulndb] Downloading artifact...	repo="mirror.gcr.io/aquasec/trivy-db:2"
23.03 MiB / 86.52 MiB [---------------->____________________________________________] 26.62% ? p/s ?43.56 MiB / 86.52 MiB [------------------------------>______________________________] 50.35% ? p/s ?66.34 MiB / 86.52 MiB [---------------------------------------------->______________] 76.68% ? p/s ?86.52 MiB / 86.52 MiB [--------------------------------------------->] 100.00% 105.81 MiB p/s ETA 0s86.52 MiB / 86.52 MiB [--------------------------------------------->] 100.00% 105.81 MiB p/s ETA 0s86.52 MiB / 86.52 MiB [--------------------------------------------->] 100.00% 105.81 MiB p/s ETA 0s86.52 MiB / 86.52 MiB [---------------------------------------------->] 100.00% 98.99 MiB p/s ETA 0s86.52 MiB / 86.52 MiB [---------------------------------------------->] 100.00% 9

In [5]:
import json
import pandas as pd
from IPython.display import display

# ── Parse Trivy JSON into a table ──
with open('./report/trivy_results.json', 'r') as f:
    trivy_data = json.load(f)

vulnerabilities = []
for result in trivy_data.get('Results', []):
    for vuln in result.get('Vulnerabilities', []):
        vulnerabilities.append({
            'CVE': vuln.get('VulnerabilityID', 'N/A'),
            'Package': vuln.get('PkgName', 'N/A'),
            'Installed': vuln.get('InstalledVersion', 'N/A'),
            'Fixed': vuln.get('FixedVersion', 'N/A'),
            'Severity': vuln.get('Severity', 'N/A'),
            'Title': vuln.get('Title', 'N/A')[:80],
        })

df_trivy = pd.DataFrame(vulnerabilities)
print(f'\n🚨 Total CVEs found: {len(df_trivy)}')
print(f'\nBy Severity:')
print(df_trivy['Severity'].value_counts().to_string())
print()
display(df_trivy)


🚨 Total CVEs found: 12

By Severity:
Severity
HIGH        11
CRITICAL     1



,CVE,Package,Installed,Fixed,Severity,Title
0,CVE-2023-30861,Flask,2.0.1,"2.3.2, 2.2.5",HIGH,flask: Possible disclosure of permanent sessio...
1,CVE-2020-14343,PyYAML,5.3.1,5.4,CRITICAL,PyYAML: incomplete fix for CVE-2020-1747
2,CVE-2023-25577,Werkzeug,2.0.1,2.2.3,HIGH,python-werkzeug: high resource usage when pars...
3,CVE-2024-34069,Werkzeug,2.0.1,3.0.3,HIGH,python-werkzeug: user may execute code on a de...
4,CVE-2023-37920,certifi,2021.5.30,2023.7.22,HIGH,python-certifi: Removal of e-Tugra root certif...
5,CVE-2023-0286,cryptography,3.3.2,39.0.1,HIGH,openssl: X.400 address type confusion in X.509...
6,CVE-2023-50782,cryptography,3.3.2,42.0.0,HIGH,python-cryptography: Bleichenbacher timing ora...
7,CVE-2026-26007,cryptography,3.3.2,46.0.5,HIGH,cryptography: cryptography Subgroup Attack Due...
8,CVE-2023-43804,urllib3,1.26.5,"2.0.6, 1.26.17",HIGH,python-urllib3: Cookie request header isn't st...
9,CVE-2025-66418,urllib3,1.26.5,2.6.0,HIGH,urllib3: urllib3: Unbounded decompression chai...


---
## Step 3 — Generate Call Graph

In [6]:
sys.path.insert(0, str(Path.cwd()))
sys.argv = ['call_graph_generator.py', TARGET_FOLDER]

import call_graph_generator  # Runs generate_call_graph on import
print('Call graph saved to ./report/callgraph.json')

[*] Analyzing 1 Python file(s)
    📄 app.py  (module: sample1.app)

✅ Call graph saved to ./report/callgraph.json
   15 functions, 50 call edges

📈 CALL GRAPH  (function → APIs it calls)

  sample1.app.CustomJSONEncoder.default
    └─▶ default
    └─▶ isinstance
    └─▶ list
    └─▶ super

  sample1.app.cause_error
    └─▶ app.route

  sample1.app.debug_info
    └─▶ app.route
    └─▶ dict

  sample1.app.delete_user
    └─▶ app.route
    └─▶ conn.close
    └─▶ conn.commit
    └─▶ conn.cursor
    └─▶ cursor.execute
    └─▶ sqlite3.connect

  sample1.app.download
    └─▶ app.route
    └─▶ flask.request.args.get
    └─▶ flask.send_file

  sample1.app.greet
    └─▶ app.route
    └─▶ flask.Markup
    └─▶ flask.escape
    └─▶ flask.request.args.get

  sample1.app.hash_password
    └─▶ app.route
    └─▶ flask.request.args.get
    └─▶ hashlib.md5
    └─▶ hexdigest
    └─▶ password.encode

  sample1.app.home
    └─▶ app.route

  sample1.app.init_db
    └─▶ conn.close
    └─▶ conn.commit
    └─▶ 

In [7]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'groq'], check=True, capture_output=True)
print('✅ Groq SDK installed')

✅ Groq SDK installed


---
## Step 4 — Reachability Analysis (Groq LLM)

In [8]:
!pip install groq -q

In [9]:
os.environ['TRIVY_PATH'] = './report/trivy_results.json'
os.environ['CALLGRAPH_PATH'] = './report/callgraph.json'
os.environ['OUTPUT_PATH'] = './report/reachable_vulns.json'

from reachability_analysis import main as run_reachability

run_reachability()

Loaded 12 vulnerabilities from Trivy.
Call graph keys: ['call_graph', 'import_map', 'function_info', 'statistics']
Prompt length: 8171 chars

REACHABLE VULNERABILITIES (used in code)
Reason: The call graph shows imports and calls corresponding to the vulnerable distributions 'Flask' and 'PyYAML'.

  • CVE-2020-14343  (PyYAML 5.3.1, CRITICAL)
  • CVE-2023-30861  (Flask 2.0.1, HIGH)

Total: 2 reachable out of 12 vulnerabilities

Results written to report/reachable_vulns.json


---
## Step 5 — Fix Reachable Vulnerabilities

In [10]:
os.environ['REACHABLE_VULNS_PATH'] = './report/reachable_vulns.json'
os.environ['REQUIREMENTS_PATH'] = f'{TARGET_FOLDER}/requirements.txt'
os.environ['APP_PATH'] = f'{TARGET_FOLDER}/app.py'
os.environ['REQUIREMENTS_FIXED_PATH'] = f'{TARGET_FOLDER}_reachable_fixed/requirements_fixed.txt'
os.environ['APP_FIXED_PATH'] = f'{TARGET_FOLDER}_reachable_fixed/app_fixed.py'

# Need to override sys.argv for fix_reachable_vuls
sys.argv = ['fix_reachable_vuls.py', TARGET_FOLDER]

from fix_reachable_vuls import main as run_fix_reachable

if Path('./report/reachable_vulns.json').exists():
    data = __import__('json').load(open('./report/reachable_vulns.json'))
    if data.get('used_vulnerability_ids'):
        run_fix_reachable()
    else:
        print('No reachable vulnerabilities to fix. Skipping.')
else:
    print('Reachable vulns file not found. Skipping.')

Loaded 2 reachable CVE(s):
  - CVE-2020-14343  (PyYAML 5.3.1 -> 5.4, CRITICAL)
  - CVE-2023-30861  (Flask 2.0.1 -> 2.3.2, 2.2.5, HIGH)

Prompt length: 10580 chars
Calling LLM to generate fixes...

BREAKING CHANGE ANALYSIS
The following breaking changes were detected:
- Flask 2.3.2: Removed `@before_first_request` decorator. Replaced with `@app.before_request`.
- Flask 2.3.2: Removed `flask.escape` and `flask.Markup`. Replaced with `markupsafe.escape` and `markupsafe.Markup`.
- Flask 2.3.2: Changed `JSONEncoder` API. Replaced with `markupsafe`-compatible implementation.
- PyYAML 5.4: No breaking changes detected.

Written: sample1_reachable_fixed/requirements_fixed.txt
Written: sample1_reachable_fixed/app_fixed.py

Done.


---
## Step 6 — Install Bandit and Run Static Analysis

In [11]:
!pip install bandit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.7/134.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.0 MB/s eta 0:00:00


In [12]:
from bandit import run_bandit

# Run bandit on app.py
app_file = Path(TARGET_FOLDER) / 'app.py'

print(f'Running Bandit on {app_file}\n')
run_bandit(str(app_file), output_dir='report')
print('\n✅ Bandit report saved to report/')


Running Bandit on sample1/app.py:

  app: 8 issues (HIGH: 3, MEDIUM: 5)

  FINDINGS (8 issues)

  [HIGH] B602 -- CWE-78
  subprocess call with shell=True identified, security issue.
  Line 125: 124         # Command injection vulnerability
125         result = subprocess.check_output(f'ping -c 1 {host}', shell=True)
126         return f'<pre>{result.decode()}</pre>'
  ----------------------------------------------------------------------

  [HIGH] B324 -- CWE-327
  Use of weak MD5 hash for security. Consider usedforsecurity=False
  Line 155: 154     # Using weak MD5 hash
155     hashed = hashlib.md5(password.encode()).hexdigest()
156     return f'Hashed password (MD5): {hashed}'
  ----------------------------------------------------------------------

  [HIGH] B201 -- CWE-94
  A Flask app appears to be run with debug=True, which exposes the Werkzeug debugger and allows the execution of arbitrary code.
  Line 211: 210     # VULNERABILITY 12: Debug mode in production
211     app.run(deb

---
## Step 7 — Fix Exploitable Vulnerabilities (Bandit Findings)

In [13]:
# Bandit saves as report/<basename>_bandit_report.json
bandit_base = app_file.stem  # 'app' or 'app_fixed'
bandit_report = f'report/{bandit_base}_bandit_report.json'

os.environ['BANDIT_REPORT_PATH'] = bandit_report
os.environ['APP_PATH'] = str(app_file)
os.environ['APP_FIXED_PATH'] = f'{TARGET_FOLDER}/app_exploitable_fixed.py'

sys.argv = ['fix_exploitable_vuls.py', TARGET_FOLDER]

from fix_exploitable_vuls import main as run_fix_exploitable

if Path(bandit_report).exists():
    run_fix_exploitable()
else:
    print(f'Bandit report {bandit_report} not found. Skipping.')

Filtered 8 HIGH/CRITICAL finding(s) out of 8 total from Bandit report:
  - B704 [MEDIUM] CWE-79: Potential XSS with ``flask.Markup`` detected. Do not use ``Markup`` on untrusted data.  (line 117)
  - B602 [HIGH] CWE-78: subprocess call with shell=True identified, security issue.  (line 125)
  - B301 [MEDIUM] CWE-502: Pickle and modules that wrap it can be unsafe when used to deserialize untrusted data, possible security issue.  (line 147)
  - B324 [HIGH] CWE-327: Use of weak MD5 hash for security. Consider usedforsecurity=False  (line 155)
  - B506 [MEDIUM] CWE-20: Use of unsafe yaml load. Allows instantiation of arbitrary objects. Consider yaml.safe_load().  (line 163)
  - B608 [MEDIUM] CWE-89: Possible SQL injection vector through string-based query construction.  (line 181)
  - B201 [HIGH] CWE-94: A Flask app appears to be run with debug=True, which exposes the Werkzeug debugger and allows the execution of arbitrary code.  (line 211)
  - B104 [MEDIUM] CWE-605: Possible binding to al

---
## Summary

Pipeline complete. Outputs:

| File | Description |
|------|-------------|
| `report/trivy_results.json` | Trivy CVE scan (JSON) |
| `report/callgraph.json` | Call graph for reachability |
| `report/reachable_vulns.json` | Reachable CVE IDs (LLM) |
| `sample1/requirements_fixed.txt` | Patched dependencies |
| `sample1/app_fixed.py` | CVE-fixed application |
| `report/app_bandit_report.json` | Bandit SAST findings |
| `sample1/app_exploitable_fixed.py` | SAST-fixed application |